# 실시간 문장 모호성 특징 추출기 (Real-time Sentence Ambiguity Feature Extractor)

본 노트북은 VLM 다중 이미지 순서 정렬 파이프라인 아키텍처의 **Phase 1(통사 구조 분류)** 및 **Phase 2(경량 의미론적 특징 추출)** 설계에 맞추어, 임의의 입력 문장에 대한 문법적 특징을 실시간으로 분석하고 추출합니다.

### 업데이트 사항 (공백 정제 및 단어 추출 추가):
- **공백 전처리 (`clean_sentence`)**: 문장 앞뒤의 빈칸 제거 및 단어 사이의 중복 공백(2칸 이상)을 1칸으로 정리하여 SpaCy 파서의 오탐율을 대폭 낮춥니다.
- **단어 리스트 추출 (`Subject_Words`, `Predicate_Words`)**: 단순 개수 카운트 외에 실제 어떤 단어가 주어이고 서술어인지 콤마로 구분된 문자열로 추출합니다.

In [ ]:
# 1. 환경 설정 및 SpaCy 모델 로드
try:
    import spacy
except ImportError:
    import subprocess
    import sys
    print("SpaCy 패키지가 없어 설치를 진행합니다...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    import spacy
import re

# 영어 경량 의존성 구문 분석 모델 로드
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("en_core_web_sm 모델 다운로드 중...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("SpaCy 환경 로드 완료!")

## 2. 노이즈 정제 및 특징 추출 함수 정의

텍스트 노이즈(중복 공백 등)를 사전에 제거하는 `clean_sentence` 필터와, 추출된 단어 목록까지 포착하는 `extract_ambiguity_features` 함수를 정의합니다.

In [ ]:
def clean_sentence(text):
    """
    문장 맨 앞/뒤의 무의미한 빈칸과 단어 사이의 중복 공백(2칸 이상)을 제거하여
    구문 파서(SpaCy)의 오탐지 확률을 방지합니다.
    """
    if not isinstance(text, str):
        return ""
    # 앞뒤 공백 제거 및 연속된 공백을 1칸으로 변환
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def extract_ambiguity_features(sentence):
    """
    임의의 영문 문장을 정제한 후,
    통사 유형(Partition), 서술어/주어 개수 및 실제 매칭 단어를 반환합니다.
    """
    # 1. 텍스트 클리닝 필터 적용
    cleaned_sentence = clean_sentence(sentence)
    
    if not cleaned_sentence:
        return {
            "Partition": "Type-1",
            "Pred_Count": 0,
            "Subj_Count": 0,
            "Subject_Words": "",
            "Predicate_Words": ""
        }
    
    doc = nlp(cleaned_sentence)
    
    # 2. 통사 구조 분류 (Type-1, 2, 3)
    has_subordinate_clause = False  # advcl (부사절 종속) 또는 ccomp (절 보충) 유무
    has_parallel_clause = False     # conj (등위절 결합) 동사/조동사 유무
    
    for token in doc:
        if token.dep_ in {"advcl", "ccomp"}:
            has_subordinate_clause = True
        if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
            has_parallel_clause = True
            
    if not has_subordinate_clause and not has_parallel_clause:
        partition = "Type-1"
    elif has_subordinate_clause:
        partition = "Type-2"
    else:
        partition = "Type-3"
        
    # 3. 주어(Subject) 및 서술어(Predicate) 단어 및 개수 추출
    subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
    subjs = [t.text for t in doc if t.dep_ in subj_deps]
    preds = [t.text for t in doc if t.pos_ in {"VERB", "AUX"}]
    
    return {
        "Partition": partition,
        "Pred_Count": len(preds),
        "Subj_Count": len(subjs),
        "Subject_Words": ", ".join(subjs),
        "Predicate_Words": ", ".join(preds)
    }

## 3. 실시간 단일 문장 테스트

아래 셀에서 공백 노이즈가 섞인 비정형 문장을 입력하고, 전처리 후 실시간으로 주어와 동사 단어들이 올바르게 감지되는지 확인합니다.

In [ ]:
# 테스트 문장 리스트 (의도적으로 앞뒤 공백 및 더블 스페이스 삽입)
test_sentences = [
    "   A dog   jumps  over the fence.   ",  # 공백 노이즈 포함
    "the woman sits in green kayak and get in the water holding the row.",  # 소문자 시작
    "While the man is cutting a paper, the woman writes on a whiteboard.",
    "The boy kicks the ball, runs to the goalpost, and celebrates."
]

print("=== 테스트 문장 분석 결과 ===")
for s in test_sentences:
    res = extract_ambiguity_features(s)
    print(f"\n원본 문장: \"{s}\"")
    print(f" -> 정제 후  : \"{clean_sentence(s)}\"")
    print(f" -> Partition: {res['Partition']}")
    print(f" -> 주어 ({res['Subj_Count']}개)  : [{res['Subject_Words']}]")
    print(f" -> 서술어 ({res['Pred_Count']}개): [{res['Predicate_Words']}]")

### 사용자 맞춤 입력 테스트
아래 문자열을 자유롭게 수정해서 테스트해 보세요!

In [ ]:
my_sentence = "   First  the camera   zooms  in , then the scene   shifts slightly  left.   "
features = extract_ambiguity_features(my_sentence)

print(f"입력 문장: \"{my_sentence}\"")
print(f"정제 문장: \"{clean_sentence(my_sentence)}\"")
print("\n--- 분석 결과 ---")
for k, v in features.items():
    print(f"{k:<15}: {v}")

## 4. 실시간 대용량 배치 처리 예제 (`nlp.pipe` 사용)

대량의 문장 리스트가 인풋으로 들어올 때 배치 처리를 통해 속도를 극대화(1ms급)하며 전처리 및 추출하는 최적화 코드입니다.

In [ ]:
def extract_features_batch(sentences, batch_size=256):
    """
    배치 연산을 통해 대량의 문장 데이터를 정제하고 특징을 가공합니다.
    """
    # 1. 전체 문장 1차 정제 필터 적용
    cleaned_sentences = [clean_sentence(s) for s in sentences]
    
    results = []
    subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
    
    # 2. nlp.pipe를 이용한 일괄 처리
    for doc in nlp.pipe(cleaned_sentences, batch_size=batch_size):
        if not doc.text.strip():
            results.append({
                "Partition": "Type-1",
                "Pred_Count": 0,
                "Subj_Count": 0,
                "Subject_Words": "",
                "Predicate_Words": ""
            })
            continue
            
        has_subordinate_clause = False
        has_parallel_clause = False
        
        for token in doc:
            if token.dep_ in {"advcl", "ccomp"}:
                has_subordinate_clause = True
            if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
                has_parallel_clause = True
                
        if not has_subordinate_clause and not has_parallel_clause:
            partition = "Type-1"
        elif has_subordinate_clause:
            partition = "Type-2"
        else:
            partition = "Type-3"
            
        subjs = [t.text for t in doc if t.dep_ in subj_deps]
        preds = [t.text for t in doc if t.pos_ in {"VERB", "AUX"}]
        
        results.append({
            "Partition": partition,
            "Pred_Count": len(preds),
            "Subj_Count": len(subjs),
            "Subject_Words": ", ".join(subjs),
            "Predicate_Words": ", ".join(preds)
        })
    return results

# 배치 테스트 실행
many_sentences = [
    "   A skater   slips on the ice.   ",
    "The chef chops onions and mixes them with garlic.",
    "   While the engine starts, the pilot   checks the instruments.   ",
    "The singer holds the microphone, walks to the stage, and sings."
] * 10  # 40개 문장 생성

batch_results = extract_features_batch(many_sentences)
print(f"배치 처리 완료! 총 {len(batch_results)}개 중 최초 4개 샘플:")
for i in range(4):
    print(f"\n샘플 {i+1}: \"{many_sentences[i].strip()}\"")
    print(f" -> 결과: {batch_results[i]}")